In [0]:
import pyspark
from pyspark.sql.types import *
import pyspark.sql.functions as f
from pyspark.sql import SparkSession, Row, DataFrame
from pyspark.sql.window import Window
from pyspark.sql.functions import col
from pyspark.sql.functions import when
from typing import Union, Optional, List
from dataclasses import dataclass
from pyspark.sql.types import IntegerType
from functools import reduce
from datetime import timedelta
from pyspark.sql.functions import broadcast
from pyspark.sql.functions import when, lit

#ignore strange depreciation warnings
from warnings import simplefilter 
simplefilter(action='ignore', category=DeprecationWarning)

In [0]:
# Pre-pivoted closed loop data pulled from closed_loop_campaign_summary notebook
closed_loop_prepivot = spark.read.option("header", "true").option("inferSchema", "true").csv('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/closed_loop_summary_tab_INTERMEDIATE.csv')
closed_loop_prepivot = closed_loop_prepivot.filter(f.col('camp_start_date') >= '2023-01-01')
closed_loop_prepivot.display()

# Metadata pulls from KPM
mmci = spark.read.parquet(f'abfss://data@sa8451midsrprd.dfs.core.windows.net/media_meas_campaign_info_v')
mda = spark.read.parquet(f'abfss://measure@sa8451camprd.dfs.core.windows.net/dashboard/campaign/version=v2/source=azure')
points_detail = (spark.read.parquet(f'abfss://data@sa8451kemprd.dfs.core.windows.net/pls_points_v2/'))
mmoi= spark.read.parquet(f'abfss://landingzone@sa8451entlakegrnprd.dfs.core.windows.net/mart/comms/prd/measurement/MEDIA_MEAS_OFFER_INFO')
new_th = spark.read.parquet(f'abfss://landingzone@sa8451entlakegrnprd.dfs.core.windows.net/mart/comms/prd/measurement/TARGET_HISTORY')
old_th = spark.read.parquet(f'abfss://landingzone@sa8451entlakegrnprd.dfs.core.windows.net/mart/comms/prd/measurement/bullseye/TARGET_HISTORY_FULL_20251015/').withColumnRenamed('ehhn', 'hshd_code').select('TARGET_ID', 'HSHD_CODE', 'PRIORITY', 'TEST_CONTROL_ID', 'OFFER_ID', 'DECILE', 'SCORE')
target_history = new_th.union(old_th)
mhtv = spark.read.parquet(f'abfss://data@sa8451midsrprd.dfs.core.windows.net/media_hist_revamped')
redemptions = spark.read.parquet(f'abfss://acds@sa8451posprd.dfs.core.windows.net/transaction_coupon_fct')
downloads = spark.read.parquet(f'abfss://measure@sa8451camprd.dfs.core.windows.net/intermediate/engagements/coupon_downloads/')

#### OL campaigns where redemption cost is $ and not fuel points
- For offers where redemption cost is dollars off rather than fuel points, redemption cost in mhtv will not be 0 or null. We would have to handle some of these edge case campaigns separately before we bring in the fuel point ROAS automation. 

In [0]:
# Identify OL campaigns. All offers involvoing fuel points should already be automated
# into absolute methodology. However, we need to handle where pg_master has a "Save $x.xx on " 
# offers, which are dollars and not fuel points.

# also edge case campaign id 148260 which should be an OL TDC campaign but is not named as such
regex_pattern = r"\b(OL|OPEN LOOP|DOLLAR OFF)\b"

kpf_closed_loop_summary_tab_OL = closed_loop_prepivot.filter(
    (
        (f.upper(f.col('project_name')).rlike(regex_pattern)) &
        (f.year(f.col('camp_start_date')) >= 2024)
    ) | (f.col("campaign_id") == "148260")
)

kpf_closed_loop_summary_tab_OL.display()

In [0]:
# Make new working cost based on channel type
kpf_closed_loop_summary_tab_OL_final = kpf_closed_loop_summary_tab_OL.withColumn(
    "multiplier",
    when(col("channel") == "Display Ad", lit(0.3520))
    .when(col("channel") == "Email Module", lit(0.015))
    .when(col("channel") == "Pandora", lit(0.741))
    .when(col("channel") == "Pinterest", lit(0.663))
    .when(col("channel") == "Pre-Roll Video", lit(0.3960))
    .when(col("channel") == "Push Notifications", lit(0.038))
    .when(col("channel") == "Roku", lit(0.90))
    .when(col("channel") == "Native", lit(1))
    .when(col("channel") == "Single Subject Email", lit(0.3390))
    .when(col("channel") == "Targeted Digital Coupon", lit(0.141))
    .otherwise(lit(0))
).withColumn(
    "working_cost",
    col("camp_cost") * col("multiplier")
)
kpf_closed_loop_summary_tab_OL_final.display()

In [0]:
# adj_total_cost = working_cost + camp cost
# abs_total_cost = working_cost + redemption_cost
# redemption_cost = abs_total_cost - working_cost. redemption_cost AKA redemption $ amount

group_cols = ["campaign_id", "project_name"]

kpf_closed_loop_summary_tab_OL_agg = (kpf_closed_loop_summary_tab_OL_final
    .filter((f.col("total_redemptions_cost").isNotNull()) & (f.col("total_redemptions_cost") != 0))
    .groupBy(group_cols)
    .agg(
        f.sum("total_redemptions_cost").alias("redemption_cost"),
        f.sum("camp_cost").alias("camp_cost"),
        f.sum("working_cost").alias("working_cost"),
        f.sum("sales_uplift_total").alias("sales_uplift_total"),
        f.sum("sales_test_total").alias("sales_test_total")
    )
)

# 3. Perform Calculations on the Aggregated Totals
# This ensures ROAS is calculated on the total campaign volume
kpf_closed_loop_summary_tab_OL_calc = (kpf_closed_loop_summary_tab_OL_agg
    .withColumn("adj_total_cost", f.col("working_cost") + f.col("camp_cost"))
    .withColumn("abs_total_cost", f.col("working_cost") + f.col("redemption_cost"))
    .withColumn('adj_sales_uplift', f.col("sales_uplift_total").cast('Integer'))
    .withColumn('adj_sales_total', f.col("sales_test_total").cast('Integer'))
    .withColumn("abs_sales_uplift_earned", f.round(f.col("sales_uplift_total") - f.col("redemption_cost"), 2))
    .withColumn("abs_sales_test_earned", f.round(f.col("sales_test_total") - f.col("redemption_cost"), 2))
    .withColumn("adj_iroas", f.round(f.col("sales_uplift_total") / f.col("adj_total_cost"), 2))
    .withColumn("abs_iroas", f.round(f.col("abs_sales_uplift_earned") / f.col("abs_total_cost"), 2))
    .withColumn("adj_aroas", f.round(f.col("sales_test_total") / f.col("adj_total_cost"), 2))
    .withColumn("abs_aroas", f.round(f.col("abs_sales_test_earned") / f.col("abs_total_cost"), 2))
)

kpf_closed_loop_summary_tab_OL_calc.select(
    "campaign_id", "project_name", "redemption_cost", "camp_cost", 
    "working_cost", "adj_total_cost", "abs_total_cost", "abs_sales_uplift_earned", 
    "abs_sales_test_earned", "adj_iroas", "abs_iroas", "adj_aroas", "abs_aroas"
).display()

In [0]:
kpf_closed_loop_summary_tab_OL_output = kpf_closed_loop_summary_tab_OL_calc \
    .withColumnRenamed("abs_sales_uplift_earned", "abs_sales_uplift") \
    .withColumnRenamed("redemption_cost", "new_redemption_cost") \
    .select(
        "campaign_id", "working_cost", "new_redemption_cost", "adj_sales_uplift", "abs_sales_uplift", "abs_sales_test_earned", "adj_iroas", "abs_iroas", "adj_aroas", "abs_aroas", "adj_total_cost", "abs_total_cost"
    )

# Make negative #s to 0, since we do not report negative #s.
cols_to_zero_neg = [
    "working_cost", "adj_sales_uplift", "abs_sales_uplift", "adj_iroas", "abs_iroas", "adj_aroas", "abs_aroas", "adj_total_cost", "abs_total_cost", "abs_sales_test_earned"
]

for c in cols_to_zero_neg:
    kpf_closed_loop_summary_tab_OL_output = kpf_closed_loop_summary_tab_OL_output.withColumn(c, f.when(f.col(c) < 0, 0).otherwise(f.col(c)))

kpf_closed_loop_summary_tab_OL_output.display()

In [0]:
kpf_closed_loop_summary_tab_OL_output.coalesce(1).write.mode("overwrite").parquet('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_OL_dollar_off')